# 8.16 — Language-model evaluation

Language-model metrics are lenses, not verdicts: likelihood, overlap precision, overlap recall, and alignment reward different virtues.

Perplexity measures surprise, BLEU measures candidate overlap precision, ROUGE measures reference coverage, and METEOR weights precision and recall. The same output can rank differently under different lenses.

Save a copy to Drive to edit.

In [ ]:

import math
import random
from collections import Counter
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 713
random.seed(SEED)
np.random.seed(SEED)


def softmax(values):
    arr = np.array(values, dtype=float)
    shifted = arr - np.max(arr)
    exp = np.exp(shifted)
    return exp / exp.sum()


def accuracy(pred, gold):
    return sum(int(a == b) for a, b in zip(pred, gold)) / max(1, len(gold))


def show_table(rows, headers):
    widths = [len(str(h)) for h in headers]
    for row in rows:
        for i, value in enumerate(row):
            widths[i] = max(widths[i], len(str(value)))
    print(" | ".join(str(h).ljust(widths[i]) for i, h in enumerate(headers)))
    print("-+-".join("-" * w for w in widths))
    for row in rows:
        print(" | ".join(str(value).ljust(widths[i]) for i, value in enumerate(row)))


def make_sequence_ladder(topic):
    if topic == "8.13":
        return [
            {"name": "D1 reversal", "sequences": [["dog", "bites"], ["bites", "dog"]], "labels": [1, 0], "max_pos": 2},
            {"name": "D2 clean order", "sequences": [["a", "then", "b"], ["b", "then", "a"], ["red", "before", "blue"], ["blue", "before", "red"], ["left", "right"]], "labels": [1, 0, 1, 0, 1], "max_pos": 5},
            {"name": "D3 agreement", "sequences": [["dogs", "chase", "cat"], ["cat", "chase", "dogs"], ["he", "likes", "tea"], ["tea", "likes", "he"], ["birds", "fly", "today"]], "labels": [1, 0, 1, 0, 1], "max_pos": 8},
            {"name": "D4 pair tasks", "sequences": [["subject", "verb", "object", "now"], ["object", "verb", "subject", "now"], ["query", "key", "value", "end"], ["value", "key", "query", "end"], ["first", "middle", "last", "stop"]], "labels": [1, 0, 1, 0, 1], "max_pos": 10},
            {"name": "D5 beyond learned", "sequences": [["start"] + ["filler"] * n + ["end"] for n in [4, 8, 12, 16, 20]], "labels": [1, 1, 1, 1, 1], "max_pos": 22},
        ]
    if topic == "8.14":
        return [
            {"name": "D1 mask toy", "T": 32, "w": 2, "evidence": [(19, 0)], "global_tokens": []},
            {"name": "D2 local deps", "T": 24, "w": 2, "evidence": [(4, 3), (8, 6), (12, 11), (16, 15), (20, 18)], "global_tokens": []},
            {"name": "D3 global random", "T": 40, "w": 2, "evidence": [(30, 0), (35, 1), (21, 18), (10, 8), (39, 3)], "global_tokens": [0, 1]},
            {"name": "D4 document snippets", "T": 64, "w": 3, "evidence": [(50, 2), (60, 4), (32, 31), (12, 10), (45, 5)], "global_tokens": [0, 2, 4]},
            {"name": "D5 far evidence", "T": 96, "w": 3, "evidence": [(90, 0), (88, 6), (70, 9), (44, 43), (30, 27)], "global_tokens": [0, 6, 9]},
        ]
    if topic == "8.15":
        return [
            {"name": "D1 logits", "rows": [[3.0, 2.0, 0.5]], "target": ["A"]},
            {"name": "D2 clean rows", "rows": [[3, 1, 0], [0, 3, 1], [1, 0, 3], [4, 2, 0], [0, 4, 2]], "target": list("ABCAB")},
            {"name": "D3 traps", "rows": [[2.1, 2.0, 0], [1.8, 1.7, 1.6], [3, 2.9, 0], [2, 2, 1.9], [0, 2.2, 2.1]], "target": list("BACCB")},
            {"name": "D4 candidates", "rows": [[2.5, 2.4, 0.1], [0.5, 2.0, 1.9], [2.2, 2.1, 2.0], [3.0, 0.2, 0.1], [0.1, 2.8, 2.7]], "target": list("ABABB")},
            {"name": "D5 long generation", "rows": [[2.0, 1.9, 0.2], [1.5, 1.4, 2.2], [2.1, 2.0, 0.4], [0.4, 2.2, 2.1], [1.9, 1.8, 0.7], [0.5, 2.4, 2.3], [2.3, 2.2, 0.2]], "target": list("ACABABB")},
        ]
    if topic == "8.16":
        return [
            {"name": "D1 lesson pair", "cand": "the cat sat", "ref": "the cat slept", "probs": [0.5, 0.25, 0.25]},
            {"name": "D2 paraphrases", "pairs": [("small cat sleeps", "small cat sleeps"), ("red bird flies", "red bird sings"), ("quick dog runs", "dog runs quickly"), ("bright moon rises", "moon rises bright"), ("blue car stops", "blue bus stops")]},
            {"name": "D3 predicate order", "pairs": [("the cat chased mouse", "the mouse chased cat"), ("doctor treats patient", "doctor helps patient"), ("team wins final", "team loses final"), ("user resets password", "password resets user"), ("model answers question", "model answers query")]},
            {"name": "D4 summaries", "pairs": [("sales rose in june", "sales rose in june after ads"), ("server failed at noon", "server recovered at noon"), ("policy changed for users", "policy changed for all users"), ("campaign reached creators", "campaign reached many creators"), ("video views increased", "video watch time increased")]},
            {"name": "D5 long refs", "pairs": [("the model answered safely with citations", "the model answered correctly with citations"), ("the summary covered three facts and omitted risk", "the summary covered three facts and named risk"), ("translation kept names but changed tense", "translation kept names and preserved tense"), ("caption mentions dog running near beach", "caption mentions dog walking near beach"), ("assistant refused unsafe request politely", "assistant refused unsafe request clearly")]},
        ]
    if topic == "8.17":
        return [
            {"name": "D1 Ada", "tokens": ["Ada", "works", "at", "OpenAI"], "tags": ["B-PER", "O", "O", "B-ORG"]},
            {"name": "D2 clean entities", "examples": [(["Ada", "joined", "OpenAI"], ["B-PER", "O", "B-ORG"]), (["Lin", "visits", "Paris"], ["B-PER", "O", "B-LOC"]), (["DeepMind", "hired", "Ilya"], ["B-ORG", "O", "B-PER"]), (["Mira", "met", "Sam"], ["B-PER", "O", "B-PER"]), (["Google", "opened", "office"], ["B-ORG", "O", "O"])]},
            {"name": "D3 adjacent padding", "examples": [(["Ada", "Bob", "spoke"], ["B-PER", "B-PER", "O"]), (["New", "York", "Labs"], ["B-LOC", "I-LOC", "B-ORG"]), (["Eve", "from", "IBM", "arrived"], ["B-PER", "O", "B-ORG", "O"]), (["Paris", "Paris", "team"], ["B-LOC", "B-ORG", "O"]), (["Zed", "Corp", "LLC"], ["B-ORG", "I-ORG", "I-ORG"])]},
            {"name": "D4 snippets", "examples": [(["Dr", "Chen", "at", "Mayo"], ["B-PER", "I-PER", "O", "B-ORG"]), (["Acme", "bought", "Beta"], ["B-ORG", "O", "B-ORG"]), (["Nina", "left", "Berlin"], ["B-PER", "O", "B-LOC"]), (["OpenAI", "San", "Francisco"], ["B-ORG", "B-LOC", "I-LOC"]), (["Raj", "from", "UN"], ["B-PER", "O", "B-ORG"])]},
            {"name": "D5 long multi", "examples": [(["Ada", "Lovelace", "worked", "with", "Babbage", "Labs"], ["B-PER", "I-PER", "O", "O", "B-ORG", "I-ORG"]), (["Maya", "from", "OpenAI", "visited", "London"], ["B-PER", "O", "B-ORG", "O", "B-LOC"]), (["IBM", "and", "Google", "met", "in", "Paris"], ["B-ORG", "O", "B-ORG", "O", "O", "B-LOC"]), (["Dr", "Lee", "joined", "Mayo", "Clinic"], ["B-PER", "I-PER", "O", "B-ORG", "I-ORG"]), (["Sam", "Altman", "visited", "New", "York"], ["B-PER", "I-PER", "O", "B-LOC", "I-LOC"])]},
        ]
    return [
        {"name": "D1 morphology", "tokens": ["walk", "walked", "walking"], "tags": ["VERB", "VERB", "VERB"]},
        {"name": "D2 clean POS", "examples": [(["dogs", "walk"], ["NOUN", "VERB"]), (["they", "walked"], ["NOUN", "VERB"]), (["walking", "helps"], ["NOUN", "VERB"]), (["red", "dogs", "run"], ["ADJ", "NOUN", "VERB"]), (["cats", "sleep"], ["NOUN", "VERB"])]},
        {"name": "D3 ambiguous", "examples": [(["walk", "home"], ["VERB", "NOUN"]), (["the", "walk"], ["DET", "NOUN"]), (["walking", "tour"], ["ADJ", "NOUN"]), (["we", "duck"], ["NOUN", "VERB"]), (["duck", "soup"], ["NOUN", "NOUN"])]},
        {"name": "D4 tagged lines", "examples": [(["bright", "birds", "sing"], ["ADJ", "NOUN", "VERB"]), (["users", "liked", "ads"], ["NOUN", "VERB", "NOUN"]), (["fast", "models", "serve"], ["ADJ", "NOUN", "VERB"]), (["the", "running", "joke"], ["DET", "ADJ", "NOUN"]), (["creators", "earned", "money"], ["NOUN", "VERB", "NOUN"])]},
        {"name": "D5 agreement", "examples": [(["the", "dogs", "were", "walking"], ["DET", "NOUN", "VERB", "VERB"]), (["a", "walk", "was", "short"], ["DET", "NOUN", "VERB", "ADJ"]), (["walking", "brands", "sell"], ["ADJ", "NOUN", "VERB"]), (["names", "ending", "ing", "confuse"], ["NOUN", "VERB", "NOUN", "VERB"]), (["past", "users", "walked"], ["ADJ", "NOUN", "VERB"])]},
    ]



def tokenize(text):
    return text.lower().split()


def perplexity(probs):
    return math.exp(-sum(math.log(p) for p in probs) / len(probs))


def bleu1(candidate, reference):
    cand = tokenize(candidate)
    ref = tokenize(reference)
    cand_counts = Counter(cand)
    ref_counts = Counter(ref)
    overlap = sum(min(cand_counts[w], ref_counts[w]) for w in cand_counts)
    return overlap / max(1, len(cand))


def rouge_recall(candidate, reference):
    cand = Counter(tokenize(candidate))
    ref = Counter(tokenize(reference))
    overlap = sum(min(cand[w], ref[w]) for w in ref)
    return overlap / max(1, sum(ref.values()))


def meteor_score(candidate, reference):
    p = bleu1(candidate, reference)
    r = rouge_recall(candidate, reference)
    if p == 0 or r == 0:
        return 0.0
    return 10 * p * r / (r + 9 * p)


def evaluate_text(candidate, reference, probs):
    return {
        "ppl": perplexity(probs),
        "bleu1": bleu1(candidate, reference),
        "rouge": rouge_recall(candidate, reference),
        "meteor": meteor_score(candidate, reference),
    }


## The concept, built once: likelihood and overlap

Perplexity is $\exp(-\frac{1}{N}\sum_t\log p(x_t|x_{<t}))$. BLEU-1 precision is $\sum_w\min(c_{cand}(w),c_{ref}(w))/\sum_w c_{cand}(w)$.

In [ ]:

metrics = evaluate_text("the cat sat", "the cat slept", [0.5, 0.25, 0.25])
print(metrics)
assert round(metrics["ppl"], 15) == 3.174802103936399
assert metrics["bleu1"] == 2 / 3


ROUGE recall rewards covered reference facts, while METEOR uses the lesson denominator $R+9P$ rather than unweighted F1. These metrics can rank systems differently.

In [ ]:

rouge = 3 / 5
meteor = 10 * 0.5 * 0.8 / (0.8 + 9 * 0.5)
model_a = {"bleu": 0.8, "rouge": 0.4}
model_b = {"bleu": 0.5, "rouge": 0.7}
print("rouge", rouge)
print("meteor", round(meteor, 6))
assert rouge == 0.6
assert round(meteor, 6) == 0.754717
assert model_a["bleu"] > model_b["bleu"]
assert model_a["rouge"] < model_b["rouge"]


## The dataset ladder

D1 is the lesson pair, then paraphrases, predicate/order changes, summaries, and D5 longer examples.

In [ ]:

ladder = make_sequence_ladder("8.16")
for rung in ladder:
    if "pairs" in rung:
        print(rung["name"], "pairs=", len(rung["pairs"]), "sample=", rung["pairs"][0])
    else:
        print(rung["name"], "candidate=", rung["cand"], "reference=", rung["ref"])


In [ ]:

rows = []
metrics_by_rung = []
for rung in ladder:
    pairs = rung.get("pairs", [(rung["cand"], rung["ref"])])
    scores = [bleu1(cand, ref) for cand, ref in pairs]
    rouges = [rouge_recall(cand, ref) for cand, ref in pairs]
    score = sum(scores) / len(scores)
    metrics_by_rung.append(score)
    rows.append([rung["name"], len(pairs), round(score, 3), round(sum(rouges) / len(rouges), 3)])
show_table(rows, ["rung", "pairs", "BLEU1", "ROUGE"])


In [ ]:

fig, axes = plt.subplots(1, 6, figsize=(15, 2.4))
for ax, rung in zip(axes[:5], ladder):
    pair = rung.get("pairs", [(rung["cand"], rung["ref"])])[0]
    cand = tokenize(pair[0])
    ref = tokenize(pair[1])
    mat = np.array([[int(a == b) for b in ref] for a in cand])
    ax.imshow(mat, cmap="Blues", aspect="auto", vmin=0, vmax=1)
    ax.set_title(rung["name"].split()[0])
    ax.set_xlabel("ref")
    ax.set_ylabel("cand")
axes[5].plot(range(1, 6), metrics_by_rung, marker="o")
axes[5].set_ylim(0, 1.05)
axes[5].set_title("BLEU1")
axes[5].set_xlabel("rung")
fig.tight_layout()


## Pitfall on D5: one metric is not a verdict

BLEU and ROUGE can reverse rankings, and perplexity cannot be compared across different tokenizers without care.

In [ ]:

d5 = ladder[-1]
bleu_scores = [bleu1(cand, ref) for cand, ref in d5["pairs"]]
rouge_scores = [rouge_recall(cand, ref) for cand, ref in d5["pairs"]]
char_token_ppl = perplexity([0.9] * 20)
word_token_ppl = perplexity([0.5] * 5)
print("BLEU table", np.round(bleu_scores, 3))
print("ROUGE table", np.round(rouge_scores, 3))
print("tokenizer-dependent PPL", round(char_token_ppl, 3), round(word_token_ppl, 3))
assert char_token_ppl != word_token_ppl
assert len(bleu_scores) == len(rouge_scores)


## Evaluate it + Practice

- Metric: BLEU1 tracked across rungs, with PPL and ROUGE as diagnostics.
- Sanity check: identical candidate/reference gives BLEU1 and ROUGE of 1.
- Ablation: remove token normalization and overlap drops on case changes.
- Failure signal: rankings change when the product goal switches from precision to recall.

Practice: add bigram BLEU to D3.

Practice: label each D5 error as tense, predicate, omission, or hallucination.